# Hold-time threshold sensitivity analysis

This notebook evaluates the effect of hold-time thresholds of 5, 10, and 30 seconds. It reuses the timestamp-reset repair policy but does not rerun windowing or model training.

Outputs are stored as CSV files so the threshold decision can be documented and compared systematically.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import gzip
import json
import math
import random
import statistics

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / 'clarkson2_preprocessing').is_dir()), HERE)
WORK_DIR = PROJECT_ROOT / 'clarkson2_preprocessing'
INPUT_DIR = WORK_DIR / 'outputs' / 'normalized_events'
OUTPUT_DIR = WORK_DIR / 'outputs' / 'hold_threshold_sensitivity'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS_SECONDS = [5.0, 10.0, 30.0]
HISTOGRAM_EDGES = [0, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1, 2, 5, 10, 30, 60, 120, 300, math.inf]
RESERVOIR_SIZE = 100_000
RANDOM_SEED = 228
input_files = sorted(INPUT_DIR.glob('*.tsv.gz'))
if not input_files:
    raise FileNotFoundError(f'No normalized files found in {INPUT_DIR}')
print('Input users:', len(input_files))
print('Thresholds:', THRESHOLDS_SECONDS)

Input users: 100
Thresholds: [5.0, 10.0, 30.0]


## Scan repaired press/release pairs

For every file, timestamp backsteps clear the active press queues and start a new segment. Hold times are counted before applying any threshold. This lets us compare thresholds without repeatedly rescanning the dataset.

Long-flight filtering is intentionally not applied here; this analysis isolates the effect of the hold-time threshold.

In [2]:
def histogram_bucket(value):
    for left, right in zip(HISTOGRAM_EDGES[:-1], HISTOGRAM_EDGES[1:]):
        if left <= value < right:
            return f'{left:g}-{right:g}' if math.isfinite(right) else f'{left:g}+'
    return 'out_of_range'

def scan_user(input_path, reservoir, reservoir_seen):
    user_id = input_path.name.removesuffix('.tsv.gz')
    stats = Counter()
    threshold_kept = Counter()
    threshold_removed = Counter()
    histogram = Counter()
    negative_key_counts = Counter()
    active = defaultdict(deque)
    previous_timestamp = None
    first_timestamp = None
    example_rows = []

    with gzip.open(input_path, 'rt', encoding='utf-8', newline='') as source:
        reader = csv.DictReader(source, delimiter='\t')
        for event_number, row in enumerate(reader, start=1):
            stats['input_events'] += 1
            try:
                timestamp = int(row['timestamp_100ns'])
                flag = row['flag']
                key = row['key']
            except (KeyError, TypeError, ValueError):
                stats['invalid_events'] += 1
                continue
            if first_timestamp is None:
                first_timestamp = timestamp
            if previous_timestamp is not None and timestamp < previous_timestamp:
                stats['timestamp_backsteps'] += 1
                stats['presses_discarded_at_backstep'] += sum(len(queue) for queue in active.values())
                active.clear()
            previous_timestamp = timestamp

            if flag == '0':
                stats['press_events'] += 1
                active[key].append((timestamp, event_number))
            elif flag == '1':
                stats['release_events'] += 1
                if not active[key]:
                    stats['unmatched_releases'] += 1
                    continue
                press_timestamp, press_event_number = active[key].popleft()
                hold_seconds = (timestamp - press_timestamp) / 10_000_000
                if hold_seconds < 0:
                    stats['negative_hold_pairs'] += 1
                    negative_key_counts[key] += 1
                    continue

                stats['nonnegative_pairs'] += 1
                histogram[histogram_bucket(hold_seconds)] += 1
                reservoir_seen += 1
                if len(reservoir) < RESERVOIR_SIZE:
                    reservoir.append(hold_seconds)
                else:
                    replacement = rng.randrange(reservoir_seen)
                    if replacement < RESERVOIR_SIZE:
                        reservoir[replacement] = hold_seconds
                for threshold in THRESHOLDS_SECONDS:
                    if hold_seconds <= threshold:
                        threshold_kept[threshold] += 1
                    else:
                        threshold_removed[threshold] += 1
                if hold_seconds > min(THRESHOLDS_SECONDS) and len(example_rows) < 100:
                    example_rows.append({
                        'user_id': user_id,
                        'event_number': event_number,
                        'press_event_number': press_event_number,
                        'key': key,
                        'press_timestamp_100ns': press_timestamp,
                        'release_timestamp_100ns': timestamp,
                        'hold_seconds': hold_seconds,
                    })
            else:
                stats['invalid_flags'] += 1

    stats['unmatched_presses'] = sum(len(queue) for queue in active.values())
    stats['user_id'] = user_id
    return dict(stats), dict(threshold_kept), dict(threshold_removed), dict(histogram), negative_key_counts, example_rows, reservoir_seen

rng = random.Random(RANDOM_SEED)
reservoir = []
reservoir_seen = 0
user_rows = []
global_kept = Counter()
global_removed = Counter()
global_histogram = Counter()
global_negative_keys = Counter()
long_hold_examples = []

for index, input_path in enumerate(input_files, start=1):
    stats, kept, removed, histogram, negative_keys, examples, reservoir_seen = scan_user(input_path, reservoir, reservoir_seen)
    for threshold in THRESHOLDS_SECONDS:
        global_kept[threshold] += kept.get(threshold, 0)
        global_removed[threshold] += removed.get(threshold, 0)
    for bucket, count in histogram.items():
        global_histogram[bucket] += count
    global_negative_keys.update(negative_keys)
    long_hold_examples.extend(examples[:20])
    for threshold in THRESHOLDS_SECONDS:
        user_rows.append({
            'user_id': stats['user_id'],
            'threshold_seconds': threshold,
            'input_events': stats.get('input_events', 0),
            'press_events': stats.get('press_events', 0),
            'release_events': stats.get('release_events', 0),
            'nonnegative_pairs': stats.get('nonnegative_pairs', 0),
            'kept_pairs': kept.get(threshold, 0),
            'removed_pairs': removed.get(threshold, 0),
            'kept_percent': kept.get(threshold, 0) / max(stats.get('nonnegative_pairs', 0), 1) * 100,
            'unmatched_presses': stats.get('unmatched_presses', 0),
            'unmatched_releases': stats.get('unmatched_releases', 0),
            'negative_hold_pairs': stats.get('negative_hold_pairs', 0),
            'timestamp_backsteps': stats.get('timestamp_backsteps', 0),
        })
    if index % 10 == 0 or index == len(input_files):
        print(f'Scanned {index}/{len(input_files)} users')

long_hold_examples = sorted(long_hold_examples, key=lambda row: row['hold_seconds'], reverse=True)[:1000]

with (OUTPUT_DIR / 'hold_threshold_sensitivity_by_user.csv').open('w', newline='', encoding='utf-8') as handle:
    fields = list(user_rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(user_rows)

global_rows = []
for threshold in THRESHOLDS_SECONDS:
    kept = global_kept[threshold]
    removed = global_removed[threshold]
    total = kept + removed
    global_rows.append({
        'threshold_seconds': threshold,
        'nonnegative_pairs': total,
        'kept_pairs': kept,
        'removed_pairs': removed,
        'kept_percent': kept / max(total, 1) * 100,
        'removed_percent': removed / max(total, 1) * 100,
    })
with (OUTPUT_DIR / 'hold_threshold_sensitivity.csv').open('w', newline='', encoding='utf-8') as handle:
    fields = list(global_rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(global_rows)

with (OUTPUT_DIR / 'hold_time_histogram.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['hold_time_bin_seconds', 'count'])
    writer.writeheader()
    for bucket in [f'{left:g}-{right:g}' if math.isfinite(right) else f'{left:g}+' for left, right in zip(HISTOGRAM_EDGES[:-1], HISTOGRAM_EDGES[1:])]:
        writer.writerow({'hold_time_bin_seconds': bucket, 'count': global_histogram.get(bucket, 0)})

sample_sorted = sorted(reservoir)
quantile_rows = []
for quantile in [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]:
    if sample_sorted:
        position = min(len(sample_sorted) - 1, int(round((len(sample_sorted) - 1) * quantile)))
        value = sample_sorted[position]
    else:
        value = None
    quantile_rows.append({'quantile': quantile, 'hold_seconds_approx': value, 'sample_size': len(sample_sorted)})
with (OUTPUT_DIR / 'hold_distribution_quantiles.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(quantile_rows[0].keys()))
    writer.writeheader()
    writer.writerows(quantile_rows)

with (OUTPUT_DIR / 'long_hold_examples.csv').open('w', newline='', encoding='utf-8') as handle:
    fields = ['user_id', 'event_number', 'press_event_number', 'key',
              'press_timestamp_100ns', 'release_timestamp_100ns', 'hold_seconds']
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(long_hold_examples)

with (OUTPUT_DIR / 'negative_hold_keys.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['key', 'negative_hold_pairs'])
    writer.writeheader()
    for key, count in global_negative_keys.most_common():
        writer.writerow({'key': key, 'negative_hold_pairs': count})

(OUTPUT_DIR / 'analysis_config.json').write_text(json.dumps({
    'thresholds_seconds': THRESHOLDS_SECONDS,
    'histogram_edges_seconds': HISTOGRAM_EDGES,
    'reservoir_size': RESERVOIR_SIZE,
    'random_seed': RANDOM_SEED,
}, indent=2), encoding='utf-8')

print('Results written to:', OUTPUT_DIR)
for row in global_rows:
    print(row)

Scanned 10/100 users


Scanned 20/100 users


Scanned 30/100 users


Scanned 40/100 users


Scanned 50/100 users


Scanned 60/100 users


Scanned 70/100 users


Scanned 80/100 users


Scanned 90/100 users


Scanned 100/100 users
Results written to: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\hold_threshold_sensitivity
{'threshold_seconds': 5.0, 'nonnegative_pairs': 10425149, 'kept_pairs': 5498480, 'removed_pairs': 4926669, 'kept_percent': 52.742459604174485, 'removed_percent': 47.25754039582552}
{'threshold_seconds': 10.0, 'nonnegative_pairs': 10425149, 'kept_pairs': 5710903, 'removed_pairs': 4714246, 'kept_percent': 54.780061177063274, 'removed_percent': 45.219938822936726}
{'threshold_seconds': 30.0, 'nonnegative_pairs': 10425149, 'kept_pairs': 6012477, 'removed_pairs': 4412672, 'kept_percent': 57.67281599524381, 'removed_percent': 42.32718400475619}


## Interpretation

Use `hold_threshold_sensitivity.csv` to select a threshold. A good threshold should remove obvious pairing/logging artifacts without discarding a large fraction of plausible human key-hold times.

The histogram and quantile files provide the distribution evidence. The per-user CSV identifies whether the long-hold problem is concentrated in a small number of users.